# EvoFlow use case: IFN-β-stimulated NK cells

This notebook asks a concrete biological question: **which signalling
mediators shape the NK-cell effector program after type I interferon
stimulation, and which of them could be knocked out to modulate it?**

It runs the complete NicheNet -> EvoFlow workflow inside the container
published with the project:

```bash
docker run --rm -p 8888:8888 ghcr.io/vergaju/evoflow:latest
```

That image already contains Python, **NicheNetPy**, **pertpy** and EvoFlow, so
no local installation is required. Open `vignettes/use_case.ipynb` inside
JupyterLab and run all cells.

## Experimental system

We use the PBMC dataset of
[Kang et al. (2018)](https://doi.org/10.1038/nbt.4042): eight donors, each with
a control sample and a sample stimulated with **IFN-β for six hours**. The
dataset is available through `pertpy`. We focus on **NK cells** and on the
**stimulated condition**, because the question is about the response to
interferon rather than the resting state. Control NK cells are kept only for
the expression comparison in section 2.

## Workflow

1. **NicheNetPy** — infer a weighted ligand -> signalling -> regulator network
   that connects the receptors expressed by NK cells to the transcriptional
   and effector programs we care about.
2. **Expression data** — quantify how broadly each network gene is expressed
   in IFN-β-stimulated NK cells (`nonzero_ratio`) and use it to weight
   capacities.
3. **EvoFlow** — run the multi-objective genetic algorithm to find small node
   sets whose removal reduces signal flow from the receptors to those targets.
4. **Interpretation** — compare the balanced, flow, cost and redundancy
   solutions and relate them to the IFN-β response.

> **First run.** NicheNetPy downloads its reference model from Zenodo (about
> 600 MB) and `pertpy` downloads the `kang_2018` dataset (about 38 MB). Both
> are cached for later runs.
>
> **Figures.** Plotting is fixed at 96 DPI with small figure sizes so the
> committed notebook stays lightweight in git.

## 0. Setup

In [ ]:
import os
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import EvoFlow

# Keep every figure small and low-resolution (the notebook is committed to git).
plt.rcParams["figure.dpi"] = 96
plt.rcParams["savefig.dpi"] = 96
plt.rcParams["figure.figsize"] = (6, 4)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)

print("EvoFlow:", EvoFlow.__version__)
print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())

## 1. Build the receptor -> target prior with NicheNetPy

NicheNet works on a **receiver-cell** perspective: we specify the receptors
that NK cells use to sense their environment, and the downstream genes whose
activity we want to explain. NicheNet then extracts the highest-weight
signalling paths that connect the corresponding ligands to those targets.

### Receptors: the IFN-β axis plus NK activation inputs

| Receptor(s)                | Why it matters for this question |
| -------------------------- | -------------------------------- |
| `IFNAR1`, `IFNAR2`         | Type I interferon receptor; the direct entry point for IFN-β (and the IFN-α family). |
| `IL2RB`, `IL2RG`, `IL15RA` | IL-2/IL-15 signalling, the main driver of NK survival, proliferation and cytotoxicity. |
| `KLRK1`                    | NKG2D, a major activating receptor for stress ligands. |
| `IL12RB1`                  | Pro-inflammatory cytokine co-stimulation that synergises with type I IFN. |

This panel deliberately keeps the prior focused: adding more receptors (for
example `KLRC1`/`KLRD1` for the NKG2A checkpoint, `NCR1` or `IL18R1`) increases
the ligand set and the inference time without changing the workflow. You can
extend `RECEPTORS` if your question needs them.

### Targets: the IFN-β response and the NK effector program

- **Immediate interferon response** — `STAT1`, `STAT2`, `IRF9` (the ISGF3
  complex), plus `IRF1` and `IRF7`.
- **NK identity and effector transcription factors** — `TBX21` (T-bet),
  `EOMES` and `PRDM1`.
- **Effector genes** — `GZMB`, `PRF1` and `IFNG`.

These targets span both the transcriptional response to IFN-β and the
functional output of NK cells, so a flow-reducing knock-out has a direct
interpretation in terms of NK activity. All targets are validated against
NicheNet's gene-regulatory network at runtime; any gene missing from the prior
is dropped with a printed message. `top_n_regulators=4` keeps the four
best-supported regulators per ligand-target pair, which keeps the network
focused and interpretable.

In [ ]:
WORKDIR = Path.cwd()
INPUT_DIR = WORKDIR / "inputs"
OUTPUT_DIR = WORKDIR / "outputs"
NICHENET_DIR = OUTPUT_DIR / "nichenet"
for directory in (INPUT_DIR, OUTPUT_DIR, NICHENET_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SAMPLE = "nk_ifnb_stim"

# Receptors that define the IFN-β and NK activation inputs of the receiver.
RECEPTORS = [
    "IFNAR1",
    "IFNAR2",
    "IL2RB",
    "IL2RG",
    "IL15RA",
    "KLRK1",
    "IL12RB1",
]

# Interferon-response, NK-lineage and effector targets.
TARGETS = [
    "STAT1",
    "STAT2",
    "IRF1",
    "IRF9",
    "IRF7",
    "TBX21",
    "EOMES",
    "PRDM1",
    "GZMB",
    "PRF1",
    "IFNG",
]

# Human-readable record of the panel (EvoFlow consumes the network and
# annotation files, not this table).
inputs = pd.concat(
    [
        pd.DataFrame({"role": "receptor", "gene": RECEPTORS}),
        pd.DataFrame({"role": "target", "gene": TARGETS}),
    ],
    ignore_index=True,
)
input_path = INPUT_DIR / f"{SAMPLE}.csv"
inputs.to_csv(input_path, index=False)
print(f"wrote {input_path}")
inputs.groupby("role")["gene"].apply(list)

Download and cache the NicheNet reference model. The ligand-target matrix and
the weighted ligand-signalling / gene-regulatory networks are published on
Zenodo. Downloads are written to a temporary `.part` file first, so an
interrupted download cannot leave a truncated cache entry behind.

In [ ]:
from urllib.request import urlopen

REFERENCE_FILES = {
    "ltf_matrix_human.pkl": (
        "https://zenodo.org/records/17061000/files/ltf_matrix_human.pkl"
    ),
    "nichenet_human.pkl": (
        "https://zenodo.org/records/17061000/files/nichenet_human.pkl"
    ),
}
reference_dir = NICHENET_DIR / "reference"
reference_dir.mkdir(parents=True, exist_ok=True)

for filename, url in REFERENCE_FILES.items():
    destination = reference_dir / filename
    if destination.exists():
        continue
    print(f"downloading {filename} ...")
    partial = destination.with_suffix(destination.suffix + ".part")
    with urlopen(url, timeout=600) as response, open(partial, "wb") as handle:
        while chunk := response.read(1 << 20):
            handle.write(chunk)
    partial.replace(destination)
    print(f"saved {destination} ({destination.stat().st_size / 1e6:.0f} MB)")

The candidate ligands are **derived from the receptors**, mirroring the
receiver-centric logic of NicheNet: any ligand in the ligand-receptor network
whose receptor is in `RECEPTORS` becomes a candidate. In this panel that
includes IFN-β itself, the IFN-α family, IL-2/IL-15, IL-12/IL-23/IL-18 and the
NKG2D/NKG2A ligands.

`get_ligand_signaling_path` then returns two weighted edge tables:

- `tf_signaling` — ligand -> receptor -> kinase/adaptor paths reaching the
  transcription factors,
- `tf_regulatory` — transcription factor -> target edges.

Together they form the prior network EvoFlow will analyse.

In [ ]:
from nichenetpy.visualization import get_ligand_signaling_path

with open(reference_dir / "ltf_matrix_human.pkl", "rb") as handle:
    ltf = pickle.load(handle)
with open(reference_dir / "nichenet_human.pkl", "rb") as handle:
    model = pickle.load(handle)

lr_sig = pd.DataFrame(model["lr_sig"]._mapping, columns=["from", "to", "weight"])
gr = pd.DataFrame(model["gr"]._mapping, columns=["from", "to", "weight"])
lr_network = pd.DataFrame(model["lr_network"]._mapping, columns=["from", "to"])

ligands = sorted(set(lr_network.loc[lr_network["to"].isin(RECEPTORS), "from"]))
gr_targets = set(gr["to"].unique())
targets = [target for target in TARGETS if target in gr_targets]
print(f"{len(ligands)} candidate ligands: {ligands}")
print(f"targets kept: {targets}")
if not ligands or not targets:
    raise RuntimeError(
        "No ligands or targets remain for the selected receptors. "
        "Adjust RECEPTORS/TARGETS."
    )

In [ ]:
tf_signaling, tf_regulatory = get_ligand_signaling_path(
    ltf["mat"],
    ligands,
    targets,
    all_ligands=ltf["row_names"],
    all_targets=ltf["col_names"],
    lr_sig=lr_sig,
    gr=gr,
    top_n_regulators=4,
    minmax_scaling=False,
)

network_df = pd.concat(
    [
        tf_signaling.assign(layer="signaling"),
        tf_regulatory.assign(layer="regulatory"),
    ],
    ignore_index=True,
)
network_nodes = set(network_df["from"]) | set(network_df["to"])
print(f"net: {len(network_df)} edges, {len(network_nodes)} nodes")
print("receptors in network:", [r for r in RECEPTORS if r in network_nodes])
print("targets in network:", [t for t in targets if t in network_nodes])
display(network_df.head())

Annotate every node so EvoFlow can report node-type proportions. Receptors are
taken from the ligand-receptor network, transcriptional regulators from the
gene-regulatory network, and everything else is treated as a signalling
mediator.

In [ ]:
receptor_genes = set(lr_network["to"])
regulator_genes = set(gr["from"])
ligand_set, target_set = set(ligands), set(targets)

rows = []
for gene in sorted(network_nodes):
    if gene in ligand_set:
        annotation = "ligand"
    elif gene in target_set:
        annotation = "target"
    elif gene in receptor_genes:
        annotation = "receptor"
    elif gene in regulator_genes:
        annotation = "transcriptional regulator"
    else:
        annotation = "signaling mediator"
    rows.append({"gene": gene, "annotation": annotation})
annotation_df = pd.DataFrame(rows)

network_path = NICHENET_DIR / f"{SAMPLE}_weighted_signaling_network.tsv"
annotation_path = NICHENET_DIR / f"{SAMPLE}_annotation_table.tsv"
network_df.to_csv(network_path, sep="\t", index=False)
annotation_df.to_csv(annotation_path, sep="\t", index=False)

print("saved", network_path.name, "and", annotation_path.name)
display(annotation_df["annotation"].value_counts())

## 2. Expression of IFN-β-stimulated NK cells

EvoFlow reduces the expression matrix to one **percolation** value per gene:
the fraction of cells in which the gene is detected (`nonzero_ratio`). That
value multiplies the edge weight, so a pathway is "open" only when its genes
are actually expressed in the cells we are modelling.

We use only the **`label == "stim"`** cells for the model, because IFN-β
rewires expression: a network calibrated on resting NK cells would not
represent the stimulated state. Control NK cells are kept for the comparison
below, but do not enter the EvoFlow capacities.

In [ ]:
import pertpy as pt

adata = pt.dt.kang_2018()
print(adata)
display(adata.obs[["cell_type", "label", "replicate"]].head())
display(adata.obs["cell_type"].value_counts())

In [ ]:
annotation_genes = set(annotation_df["gene"].astype(str)) & set(adata.var_names)
genes = sorted(annotation_genes)
print(
    f"{len(genes)} of {annotation_df['gene'].nunique()} annotated genes are "
    "measured in kang_2018"
)

nk_all = adata[adata.obs["cell_type"].astype(str) == "NK cells", genes].copy()
nk_all.obs["condition"] = nk_all.obs["label"].astype(str)
print("NK cells per condition:")
display(nk_all.obs["condition"].value_counts())

### How IFN-β changes the network genes

Before modelling, compare the detection rate of every network gene between
control and stimulated NK cells. Genes that move most are the ones whose
capacity changes when we switch from the resting to the stimulated state.

In [ ]:
values_all = (
    nk_all.X.toarray() if hasattr(nk_all.X, "toarray") else np.asarray(nk_all.X)
)
is_ctrl = (nk_all.obs["condition"] == "ctrl").to_numpy()
is_stim = (nk_all.obs["condition"] == "stim").to_numpy()

detection = pd.DataFrame(
    {
        "ctrl": (values_all[is_ctrl] > 0).mean(axis=0),
        "stim": (values_all[is_stim] > 0).mean(axis=0),
    },
    index=nk_all.var_names,
)
detection["delta"] = detection["stim"] - detection["ctrl"]
top_changed = detection.reindex(
    detection["delta"].abs().sort_values(ascending=False).head(12).index
).sort_values("delta")

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(top_changed.index, top_changed["delta"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("detection rate: stim - ctrl")
ax.set_title("Genes most changed by IFN-β")
fig.tight_layout()
plt.show()

### NK marker genes

A quick sanity check that the selected cells are NK cells and that the
stimulation produced the expected effector response.

In [ ]:
marker_genes = [g for g in ["NKG7", "GNLY", "PRF1", "GZMB"] if g in adata.var_names]
nk_markers = adata[
    adata.obs["cell_type"].astype(str) == "NK cells", marker_genes
].copy()
nk_markers.obs["condition"] = nk_markers.obs["label"].astype(str)
if marker_genes:
    sc.pl.violin(
        nk_markers, marker_genes, groupby="condition", show=False, stripplot=False
    )
    plt.gcf().set_size_inches(6, 3)
    plt.show()

Build the count matrix used by EvoFlow from the **stimulated** NK cells only.
Subsetting before densifying keeps the matrix small (the full dataset is about
25k cells x 16k genes, while this matrix is only a few thousand cells by the
network genes).

In [ ]:
is_stim = (nk_all.obs["condition"] == "stim").to_numpy()
nk_stim = nk_all[is_stim].copy()
print("stimulated NK cells:", nk_stim.n_obs, "| genes:", nk_stim.n_vars)

values = nk_stim.X.toarray() if hasattr(nk_stim.X, "toarray") else np.asarray(nk_stim.X)
expression_df = pd.DataFrame(values, index=nk_stim.obs_names, columns=nk_stim.var_names)
expression_path = INPUT_DIR / "kang_2018_nk_stim_expression.csv"
expression_df.to_csv(expression_path)
print("saved", expression_path, expression_df.shape)

## 3. Run EvoFlow on the IFN-β network

The low-level API mirrors the reference pipeline: load network, annotation and
expression, compute percolation, build the graph, assign capacities and add the
`SuperSource`/`SuperSink` topology.

`lock_targets=1` protects the original receptor nodes: they are the biological
inputs of the network, so removing them would answer a different question
("block the receptor") rather than "which intracellular mediator to target".

In [ ]:
from EvoFlow.biology import BioNetworkPreparer

preparer = BioNetworkPreparer(lock_targets=1)
preparer.load_data(
    network_path=str(network_path),
    annotation_path=str(annotation_path),
    expression_path=str(expression_path),
)
preparer.compute_percolation(stat="nonzero_ratio")
preparer.build_base_network()
preparer.update_edge_capacities(method="multiplicative")

# Keep only the receptors/targets that made it into the inferred network.
receptors = [gene for gene in RECEPTORS if gene in preparer.G]
sinks_genes = [gene for gene in TARGETS if gene in preparer.G]
print("sources:", receptors)
print("sinks:", sinks_genes)
if not receptors or not sinks_genes:
    raise RuntimeError(
        "The NicheNet network contains none of the selected receptors/targets. "
        "Inspect the annotation table and adjust RECEPTORS/TARGETS."
    )

sources, sinks = preparer.prepare_bioflow_topology(
    receptors=receptors, target_genes=sinks_genes
)
network = preparer.G
print(f"graph: {network.number_of_nodes()} nodes, {network.number_of_edges()} edges")

The genetic algorithm minimizes three objectives at once: remaining **flow**,
number of **removed nodes** (the experimental cost) and residual **SCC
redundancy**. The removal budget is set to 5% of the network, which keeps the
knock-out sets small enough to be testable.

Parameters are fixed here so the vignette is short and reproducible. In a real
analysis you would omit `population_size`/`early_stop_gen` and let the bundled
GAM choose them from the network features.

In [ ]:
from EvoFlow.core import GAOptimizer

budget = max(2, int(round(0.05 * network.number_of_nodes())))
print("removal budget:", budget)

optimizer = GAOptimizer(
    network,
    sources,
    sinks,
    max_remove=budget,
    delta_penalty=False,
    repair=True,
    dynamic=True,
    use_multiprocessing=False,  # deterministic and safe inside Jupyter
)

evoflow_dir = OUTPUT_DIR / "evoflow"
evoflow_dir.mkdir(parents=True, exist_ok=True)
previous_cwd = Path.cwd()
os.chdir(evoflow_dir)
try:
    results, logs, population, norm = optimizer.run(
        generations=200,
        population_size=50,
        early_stop_gen=20,
        guess_parameters=False,
        verbose=True,
    )
finally:
    os.chdir(previous_cwd)
print("optimization complete")

## 4. Inspect the results

Each row is one point on the Pareto front:

- **Flow** — remaining maximum flow from the receptors to the NK targets;
  lower means a stronger functional knock-out.
- **Node** — number of removed mediators (experimental cost).
- **Redundancy** — residual strongly-connected redundancy in the network.
- **Best_balanced** — the front point closest to the origin after
  per-objective normalisation.

The `removed_nodes` column is the concrete hypothesis list to follow up.

In [ ]:
rows = []
for solution, payload in results.items():
    row = {
        "solution": solution,
        "num_removed": payload["num_removed"],
        **{name: round(value, 3) for name, value in payload["fitness"].items()},
        "removed_nodes": ", ".join(payload["removed_nodes"]),
    }
    rows.append(row)
summary = pd.DataFrame(rows).set_index("solution")
display(summary)

In [ ]:
logs_df = pd.DataFrame(logs).T.reset_index(names="gen").sort_values("gen")
logs_df["gen"] = logs_df["gen"].astype(int)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, metric in zip(axes, ["Flow", "Redundancy"]):
    if metric in logs_df:
        ax.plot(logs_df["gen"], logs_df[metric], linewidth=1.2)
        ax.set_title(metric)
        ax.set_xlabel("generation")
        ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
from EvoFlow.visualization import plot_graph

final_graph = optimizer.SG.copy()
final_graph.remove_nodes_from(results["Best_balanced"]["removed_nodes"])
fig, ax = plot_graph(final_graph, filename_prefix=None, layout="spring", figsize=(6, 4))
plt.show()

### How to interpret the candidates

The targets (`STAT1`, `STAT2`, `IRF9`, `IRF1`, `IRF7`, `TBX21`, `EOMES`,
`PRDM1`, ...) and the receptors (`lock_targets=1`) are protected, so the
candidates are intracellular mediators that sit between the IFN-β / NK
activation inputs and the transcriptional response.

Because capacities are weighted by **stimulated** NK-cell detection rates, an
edge is easy to cut when either endpoint is barely expressed. This is
intentional: the algorithm is asked for interventions that matter in the
IFN-β-stimulated state, not in resting cells.

A practical follow-up is to:

1. check that the `Best_balanced` genes are expressed in the stimulated NK
   cells (the detection plot above),
2. test them for differential expression / differential detection between
   `ctrl` and `stim`,
3. prioritise mediators that are also supported by perturbation data.

## Alternative: the one-call CLI / Python API

For production runs the same pipeline is available through
`EvoFlow.cli.run_evoflow` (and the `evoflow` console script). It also handles
h5ad cell-type filtering, bulk expression files and automatic output naming:

```python
from EvoFlow.cli import run_evoflow

run_evoflow(
    network=str(network_path),
    annotation=str(annotation_path),
    expression_file=str(expression_path),
    receptors_raw="IFNAR1|IFNAR2|IL2RB|IL15RA",
    mediators_raw="STAT1,STAT2,IRF9,TBX21,EOMES",
    ligands_file="ligands.csv",   # CSV with an 'x' column of ligand genes
    targets="STAT1,STAT2,IRF9,GZMB,PRF1",  # alternative to a regulons file
    generations=2000,
)
```

Use the low-level API shown above when you already have pre-processed inputs
or want to keep the `GAOptimizer` object (for example to inspect the active
subgraph or to pickle it for later runs).

## Saving the executed notebook

The committed version of this notebook intentionally contains **no outputs**.
To publish it with results:

1. build/pull the container described in the documentation,
2. run `vignettes/use_case.ipynb` from top to bottom in JupyterLab,
3. save it (`File > Save and Checkpoint`) and commit the `.ipynb` before
   tagging the release.

All figures are already configured at 96 DPI with small figure sizes, and the
only downloaded data are cached outside the repository.